<a href="https://colab.research.google.com/github/SrilekhaM-Github/Python-web/blob/main/Blinkit_LLM_to_AgenticAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From LLM to Agentic AI — Blinkit Support Assistant

One customer problem, four levels of AI. Same model all the way through. What changes is what the model is allowed to reach.

| Level | What it can do | What it still cannot do |
|---|---|---|
| 1. LLM | Understand language | Look anything up |
| 2. RAG | Retrieve facts from policy, database, web | Take an action |
| 3. AI Agent | Choose tools and act toward a goal | Handle work that needs a team |
| 4. Agentic AI | Plan, delegate to specialists, report back | — |

**The business:** Blinkit, quick commerce. Dark stores, pickers, 10-minute delivery, item-level refunds. Not restaurants.

Files you need in [Blinkit Policy and FAQ.pdf](https://drive.google.com/file/d/12Da1uu_qSXI_zcTBLimvE3pXysSer3zk/view?usp=sharing) and [blinkit.db](https://drive.google.com/file/d/1jLHmL_bPKyYRItpfZRsRMvvlM4dC6PLb/view?usp=sharing) (11 tables, 40,000 orders, 199,000 line items, 60 cities, 444 dark stores).

# Configuration

In [1]:
# @title Step 0: Install Libraries
!pip install -q -U google-genai langchain langchain-core langchain-text-splitters langchain-google-genai deepagents tavily-python pypdf 2>&1 | grep -v "dependency resolver\|google-colab 1.0.0 requires"

# Verify every import this notebook needs, before we depend on it five cells later.
import importlib
missing = []
for mod in ["google.genai", "langchain_core", "langchain_text_splitters",
            "langchain_google_genai", "deepagents", "tavily", "pypdf"]:
    try:
        importlib.import_module(mod)
        print("OK       ", mod)
    except Exception as e:
        missing.append(mod)
        print("MISSING  ", mod, "->", type(e).__name__)

print()
print("All libraries ready." if not missing else f"Install these before continuing: {missing}")
# Colab may warn that google-auth/requests versions changed. Harmless — nothing here uses them.

google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
OK        google.genai
OK        langchain_core
OK        langchain_text_splitters
OK        langchain_google_genai
OK        deepagents
OK        tavily
OK        pypdf

All libraries ready.


In [3]:
# @title Step 1: API Keys and Model

import os, getpass
from google import genai
from IPython.display import Markdown, display

if 'GOOGLE_API_KEY' not in os.environ:
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter your Gemini API Key: ')
if 'TAVILY_API_KEY' not in os.environ:
    os.environ['TAVILY_API_KEY'] = getpass.getpass('Enter your Tavily API Key: ')

MODEL = "gemini-3.1-flash-lite" # @param {type:"string"}
client = genai.Client()
print("Ready. Model:", MODEL)

Enter your Gemini API Key: ··········
Enter your Tavily API Key: ··········
Ready. Model: gemini-3.1-flash-lite


## Get API Keys For Gemini and Tavily
##1. [Gemini API Key](https://aistudio.google.com/api-keys)
##2. [Tavily API Key](https://app.tavily.com/home)

In [4]:
# @title Step 2: Load the Data Files
import os, sqlite3

PDF_PATH = "/content/Blinkit Policy and FAQ.pdf" # @param {type:"string"}
DB_PATH  = "/content/blinkit.db" # @param {type:"string"}

for path in (PDF_PATH, DB_PATH):
    print(("FOUND    " if os.path.exists(path) else "MISSING  ") + path)

if os.path.exists(DB_PATH):
    con = sqlite3.connect(DB_PATH)
    for t in ["cities", "dark_stores", "delivery_partners", "customers", "products",
              "inventory", "orders", "order_items", "refunds", "support_tickets", "reviews"]:
        n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        print(f"  {t:20s} {n:>9,} rows")
    con.close()

FOUND    /content/Blinkit Policy and FAQ.pdf
FOUND    /content/blinkit.db
  cities                      60 rows
  dark_stores                444 rows
  delivery_partners        3,978 rows
  customers                6,000 rows
  products                   200 rows
  inventory               80,007 rows
  orders                  40,000 rows
  order_items            199,324 rows
  refunds                  7,544 rows
  support_tickets          4,771 rows
  reviews                 10,930 rows


### What is inside `blinkit.db`

| Table | What it holds | Why it matters for support |
|---|---|---|
| `cities` | 60 cities, tier, zone, monsoon flag | Late-delivery rates differ by city |
| `dark_stores` | 444 stores with pick-pack time and SKU count | A bad store shows up as refunds |
| `delivery_partners` | 3,978 riders with vehicle and rating | Rider behaviour tickets point here |
| `customers` | 6,000 customers, REGULAR or GOLD, abuse flag | Gold changes what the policy promises |
| `products` | 200 SKUs across 17 categories with MRP, perishable and weight-based flags | Refund rules differ by product type |
| `inventory` | stock per store per SKU | Stock-outs cause cancellations |
| `orders` | 40,000 orders with every fee line, promised vs actual minutes | The 10-minute promise, measured |
| `order_items` | 199,000 line items with per-item status (MISSING, DAMAGED, WEIGHT_SHORT ...) | Blinkit refunds per item, not per order |
| `refunds` | 7,544 refunds, item-level or order-level, with reject reasons | Where the money goes |
| `support_tickets` | 4,771 tickets with channel, escalation stage, CSAT | The three-stage escalation in the policy |
| `reviews` | 10,930 reviews with separate delivery and product ratings | Section 9 says these cannot be deleted |

#1. LLM - Large Language Model

The model knows language. It knows nothing about your business.

**Ask it this and watch what happens:** `Where is my refund for order 10245?`

In [5]:
# @title Step 3: Gemini — General Response, No System Prompt
from google import genai
from IPython.display import Markdown, display

query = 'Where is my refund for order 10245?' # @param {type:"string"}

client = genai.Client()

interaction = client.interactions.create(
    model=MODEL,
    input=query
)

display(Markdown(interaction.output_text))

I do not have access to your personal financial records, private account details, or specific order databases. I am an AI, not a customer service representative for a specific company.

To find the status of your refund for order #10245, please follow these steps:

1.  **Check your confirmation email:** Look for the refund confirmation message from the company you purchased from. Sometimes these emails provide a tracking link or an estimated processing time.
2.  **Log in to your account:** Go directly to the website where you placed the order and check your **"Order History"** or **"My Account"** page. The status of the refund should be listed there.
3.  **Contact the merchant:** Look for a "Contact Us" or "Customer Support" page on the website where you made the purchase. Provide them with your order number (10245) and ask for an update.
4.  **Check your bank/payment provider:** Refunds can take anywhere from **3 to 10 business days** (sometimes longer, depending on your bank) to appear on your statement after the merchant processes them.

**Important:** If you received this order number from a specific brand or store, please reach out to **their official customer service department directly.** Do not share your credit card information or private login credentials with anyone except on the store's official, secure website.

## System Prompt---LLM


You are a helpful and polite Blinkit customer-support AI assistant.

Answer customer questions using only:

1. Your existing knowledge.
2. Information explicitly provided in the conversation.

You have NO access to Blinkit's live systems, databases, customer accounts, orders, dark stores, inventory, payments, delivery tracking, policies, or external data sources.

**Never hallucinate or guess.** Do not invent order status, delivery times, refund eligibility, prices, stock availability, policies, or any customer-specific information.

If you don't have enough information to answer accurately, clearly say that you cannot verify or access that information.

Be concise, helpful, and transparent about your limitations.


In [7]:
# @title Step 4: Blinkit AI Assistant — With System Prompt
from google import genai
from IPython.display import Markdown, display

query = 'Where is my refund for order 10245?' # @param {type:"string"}

LLM_SYSTEM_PROMPT = """You are a helpful and polite Blinkit customer-support AI assistant.

Answer questions using only your existing knowledge and information provided in the conversation.

Do not claim access to live Blinkit systems, customer accounts, orders, dark stores, inventory,
payments, delivery tracking, or other real-time data.

Never hallucinate, guess, or invent information.

If a question requires information you cannot access or verify, clearly say so and provide the
most helpful alternative you can.

For simple questions about who you are, what you can do, or general topics, answer naturally
without unnecessarily mentioning your limitations.

Be concise, accurate, and transparent.
"""

client = genai.Client()

interaction = client.interactions.create(
    model=MODEL,
    input=query,
    system_instruction=LLM_SYSTEM_PROMPT,
)

display(Markdown(interaction.output_text))

I do not have access to live Blinkit systems, customer accounts, or order details, so I cannot check the status of your refund for order 10245.

To check your refund status, please use the **"Help"** or **"Support"** section within the Blinkit app. From there, you can select the specific order and view its current status or chat directly with a live support agent who can access your account details.

**What changed:** the honest version now refuses. That is better behaviour, but the customer
still has no answer. A prompt cannot create knowledge the model never had.

That gap is what RAG closes.

## Stage 1 test prompts — LLM

Paste these into the `query` box in Step 3 or Step 4.

**Works — pure language, no lookup needed**

| # | Prompt |
|---|---|
| 1 | What is the difference between a handling fee and a delivery fee? |
| 2 | Explain what a dark store is in two simple lines. |
| 3 | Write a polite apology to a customer whose milk packet arrived leaking. |
| 4 | My bill shows item total 489, surge fee 25 and GST 4.50. Break this down for me in plain language. |
| 5 | Why is GST charged on the delivery fee but not on the groceries themselves? |
| 6 | Rewrite this angry message politely: "Ten minutes my foot. Forty minutes and the eggs are broken." |
| 7 | What questions should a support agent ask before approving a missing-item refund? |
| 8 | Translate this apology into Tamil: "We are sorry your order arrived late." |
| 9 | Give me five common reasons a 10-minute grocery delivery runs late. |
| 10 | What is the difference between quick commerce and regular online grocery? |

**Breaks — the wall that RAG exists to solve**

| # | Prompt | What you will see |
|---|---|---|
| 11 | Where is my refund for order 10245? | Step 3 invents a status. Step 4 honestly refuses. Neither helps the customer. |
| 12 | Is Keerthana R. a Gold member? | No way to know. She is — but the model is guessing either way. |
| 13 | Exactly how long is Blinkit's free cancellation window? | It will guess a number. The policy says 60 seconds. |
| 14 | How many orders did we deliver late in Coimbatore? | A confident invented figure. The real answer is 165 out of 669. |

Run 11 in Step 3 and again in Step 4. The difference between a confident lie and an honest
"I cannot check that" is the whole point of a system prompt — and it still leaves the customer stuck.

#2. RAG - Retrieval Augmented Generation

Same model. Now we hand it the facts before it answers.

Three sources, three flavours of RAG:

| Flavour | Source | Answers questions like |
|---|---|---|
| Document RAG | Policy PDF | "What is the weight tolerance on loose vegetables?" |
| SQL RAG | `blinkit.db` | "Where is *my* refund?" |
| Browser RAG | Tavily web search | "Has Blinkit changed its handling fee this year?" |

## System Prompt - RAG

### Blinkit RAG AI Assistant

You are a helpful, polite, and accurate Blinkit customer-support AI assistant.

## Knowledge Sources

You can use the following sources to answer customer questions:

1. **Policy PDF** — Blinkit policies on ordering, fees, cancellation, the 10-minute promise, item-level refunds, Gold membership, escalation and account conduct.
2. **Blinkit Database** — Customers, dark stores, delivery partners, products, inventory, orders, line items, refunds, tickets and reviews.
3. **Web Search (Tavily)** — Current or external information that is not available in the connected Blinkit sources.

## How to Answer

* Use your existing knowledge for general questions.
* Use the **Policy PDF** for Blinkit policy-related questions.
* Use the **Database** for customer-specific and operational Blinkit information.
* Use **Tavily** when current web information is required.
* When a question requires multiple sources, combine the relevant information from those sources.
* Prefer retrieved source information over assumptions or general knowledge.

## Accuracy Rules

* Never hallucinate, guess, or invent information.
* Never make up order status, customer details, stock levels, prices, refunds, policies, or other data.
* Do not claim information is current unless it comes from an appropriate connected source.
* If the required information cannot be found or verified, clearly tell the customer that you cannot confirm it.
* Never claim to have retrieved or checked information that was not actually available.

## Response Style

* Be concise, clear, helpful, and customer-friendly.
* Answer the customer's question directly.
* Do not unnecessarily explain the RAG system or data sources.
* Mention the source only when it helps explain or support the answer.

**Primary goal:** Give the most accurate answer possible using the available knowledge and retrieved information. If reliable information is unavailable, be transparent rather than guessing.

## Policy and FAQ Data Source

In [8]:
# @title Step 5: Index the Policy PDF
import pypdf
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

RAG_SYSTEM_PROMPT = """You are a helpful, polite and accurate Blinkit customer-support AI assistant.

Answer using the retrieved information given to you in the prompt. Prefer it over your own knowledge.

Never hallucinate order status, customer details, prices, stock, refunds or policy rules. If the retrieved
information does not contain the answer, say clearly that you cannot confirm it.

Be concise and customer-friendly. Answer the question directly. Do not explain the retrieval system.
"""

def ask_gemini(prompt, system=RAG_SYSTEM_PROMPT):
    """One Gemini call. Used by every RAG step below."""
    return client.interactions.create(
        model=MODEL, input=prompt, system_instruction=system
    ).output_text

reader = pypdf.PdfReader(PDF_PATH)
pages  = [Document(page_content=p.extract_text() or "", metadata={"page": i + 1})
          for i, p in enumerate(reader.pages)]
chunks = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150).split_documents(pages)
policy_index = InMemoryVectorStore.from_documents(
    chunks, GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
)

print(f"{len(pages)} pages -> {len(chunks)} chunks indexed")

7 pages -> 17 chunks indexed


In [9]:
# @title Step 6: Document RAG — Ask the Policy
query = 'If I cancel my order after the store has started packing, will I get my refund?' # @param {type:"string"}

docs = policy_index.similarity_search(query, k=3)
context = "\n\n".join(d.page_content for d in docs)

answer = ask_gemini(f"Policy extract:\n{context}\n\nCustomer question: {query}")

display(Markdown(answer))
print("\nRetrieved from pages:", [d.metadata.get("page") for d in docs])

If you cancel your order after packing has started, a cancellation charge will apply (up to 100% for perishable items and up to 50% for non-perishable items), and delivery fees will not be refunded. The exact charge will be shown to you before you confirm the cancellation.


Retrieved from pages: [6, 6, 2]


## Database Source

In [10]:
# @title Step 7: SQL RAG — Ask the Database
import sqlite3, pandas as pd

query = 'Tell me the current order status of my order 10246 and which items were in it' # @param {type:"string"}

TABLES = ["cities","dark_stores","delivery_partners","customers","products","inventory",
          "orders","order_items","refunds","support_tickets","reviews"]
con = sqlite3.connect(DB_PATH)
SCHEMA = "\n".join(
    r[0] for r in con.execute(
        "SELECT sql FROM sqlite_master WHERE type='table' AND name IN (%s)"
        % ",".join("?"*len(TABLES)), TABLES)
)

sql = ask_gemini(
    f"SQLite schema:\n{SCHEMA}\n\nQuestion: {query}\n\n"
    "Return ONLY one SQLite SELECT query. No explanation, no markdown fences. "
    "Join tables when the question needs names instead of ids.",
    system="You write correct, minimal SQLite SELECT queries."
).replace("```sql", "").replace("```", "").strip()

print("Generated SQL:\n", sql, "\n")
rows = pd.read_sql(sql, con)
display(rows)

answer = ask_gemini(f"Question: {query}\n\nRows from the Blinkit database:\n{rows.to_string(index=False)}")
display(Markdown(answer))
con.close()

Generated SQL:
 SELECT o.order_status, p.product_name, oi.quantity FROM orders o JOIN order_items oi ON o.order_id = oi.order_id JOIN products p ON oi.product_id = p.product_id WHERE o.order_id = 10246; 



,order_status,product_name,quantity
0,DELIVERED,Probiotic Yogurt,1
1,DELIVERED,Papaya,1
2,DELIVERED,Cold Coffee,1
3,DELIVERED,Pepsi,1
4,DELIVERED,Filter Coffee Powder,1
5,DELIVERED,Packaged Water,1


Your order 10246 has been **DELIVERED**.

The items in your order were:
*   Probiotic Yogurt
*   Papaya
*   Cold Coffee
*   Pepsi
*   Filter Coffee Powder
*   Packaged Water

## Web Search

In [11]:
# @title Step 8: Browser RAG — Ask the Live Web (Tavily)
from tavily import TavilyClient

query = 'is there is any recent update in blinkit delivery charge' # @param {type:"string"}

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
results = tavily.search(query, max_results=3)

context = "\n\n".join(f"{r['title']}\n{r['content']}" for r in results["results"])
answer  = ask_gemini(f"Web search results:\n{context}\n\nQuestion: {query}")

display(Markdown(answer))
print("\nSources:")
for r in results["results"]:
    print(" -", r["url"])

Yes, there has been a recent update. Blinkit has dropped delivery charges in certain micro-markets across cities like Gurgaon, Bengaluru, and Mumbai to remain competitive against other quick commerce players. Please note that this change is specific to select markets.


Sources:
 - https://m.economictimes.com/tech/technology/quick-commerce-market-heats-up-blinkit-drops-delivery-fees-in-select-markets/articleshow/126590779.cms
 - https://www.storyboard18.com/brand-marketing/blinkit-zepto-instamart-raise-fees-as-quick-commerce-goes-mainstream-ws-l-99758.htm
 - https://www.linkedin.com/posts/koushik-m_so-blinkit-charges-16-for-delivery-on-orders-activity-7263896615847886848-3KfP


**Where RAG stops.** It found the refund, quoted the policy, checked the web. It told the
customer their INR 150 goodwill credit is stuck in PROCESSING and the INR 36 for the missing curd already hit the wallet.

It did not chase the bank. It did not escalate the ticket. It cannot — retrieval is read-only.

## Stage 2 test prompts — RAG

Each prompt goes in a specific step, because at this stage **you** still choose the source.

**Works — Document RAG (Step 6, the policy PDF)**

| # | Prompt | Grounded answer |
|---|---|---|
| 1 | How long does a card refund take to reach my bank account? | 3 to 5 working days; UPI 1 to 3; wallet instant |
| 2 | If I cancel after the store has started packing perishables, what do I pay? | Up to 100 percent of the item total; fees not refunded |
| 3 | My 500 g pack of tomatoes weighed 420 g. What am I owed? | Beyond the 10 percent tolerance; proportional refund with a photo on a scale |
| 4 | What are the three escalation stages and how long does each take? | 48 hours, 7 days, 30 days |
| 5 | Can the rider leave cigarettes at my door? | No; photo ID at the door, never left unattended |

**Works — SQL RAG (Step 7, blinkit.db)**

| # | Prompt | Grounded answer |
|---|---|---|
| 6 | Where is my refund for order 10245 and how late was that order? | INR 150 PROCESSING to source, INR 36 COMPLETED to wallet; 29 minutes late |
| 7 | How many refunds are still in PROCESSING and what is the total value in INR? | 717 refunds, INR 1,55,341 |
| 8 | Which city has the worst late-delivery rate? | Vijayawada, 33.0 percent of 336 delivered orders |
| 9 | How many orders has Keerthana R. placed and how many refunds has she received? | 6 orders (5 delivered, 1 failed), 3 refunds |
| 10 | Which product goes missing from bags most often? | Eggs, 41 times; then Lemon and Full Cream Milk |
| 11 | How many orders were cancelled because of stock-outs? | 685 |

**Works — Browser RAG (Step 8, Tavily)**

| # | Prompt |
|---|---|
| 12 | What is the current news on handling fees charged by quick-commerce apps in India? |
| 13 | Which quick-commerce company has the largest market share in India right now, Blinkit, Zepto or Instamart? |

**Breaks — why one retriever is not enough**

| # | Prompt | Why it fails |
|---|---|---|
| 14 | Order 10245 is 29 minutes late. Does this Gold customer qualify for the higher goodwill band? | Needs the policy AND the order in one answer. Each cell reads one source. |
| 15 | Escalate the ticket for order 10245 to the grievance officer. | Retrieval is read-only. Nothing happens. |
| 16 | Find every refund pending over 5 working days and decide which to push to wallet first. | Retrieval fetches. It does not judge or sequence. |
| 17 | Compare our late-delivery rate to what Zepto and Instamart publish. | Needs the database and the web chained together. |

Prompt 14 is the one to run live. Try it in Step 6 and it quotes policy with no order. Try it in
Step 7 and it reports the order with no policy. Then carry it into Step 10.

#3. AI Agents

An agent is **goal + reasoning + tools + memory**. We stop telling it *which* source to use and
let it decide.

The same three sources become three tools, wired into a LangChain **Deep Agent**.

In [12]:
# @title Step 9: Turn the Three Sources into Tools
import sqlite3, pandas as pd
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def policy_search(query: str) -> str:
    """Search the Blinkit Ordering, Delivery and Refunds Policy and FAQ document.
    Use for questions about rules, fees, the 10-minute promise, item-level refunds, weight
    tolerance, Gold benefits, age-restricted items or escalation. Returns the most relevant passages."""
    docs = policy_index.similarity_search(query, k=3)
    return "\n\n".join(d.page_content for d in docs)

def order_lookup(sql: str) -> str:
    """Run ONE read-only SQLite SELECT query against the Blinkit operations database and
    return the rows as text. Use for anything customer or operations specific: orders, line items,
    refunds, tickets, dark stores, inventory, delivery partners, cities. Only SELECT is allowed."""
    if not sql.strip().lower().startswith("select"):
        return "Rejected: only SELECT queries are allowed."
    try:
        with sqlite3.connect(DB_PATH) as con:
            df = pd.read_sql(sql, con)
    except Exception as e:
        return f"SQL error: {e}. Check the schema and try again."
    return "No rows found." if df.empty else df.head(25).to_string(index=False)

def web_search(query: str) -> str:
    """Search the live web for information that is not in Blinkit's own policy or database,
    such as news, competitor fees (Zepto, Swiggy Instamart) or regulatory changes."""
    r = tavily.search(query, max_results=3)
    return "\n\n".join(f"{x['title']}\n{x['content']}\n{x['url']}" for x in r["results"])

print("3 tools ready:", [f.__name__ for f in (policy_search, order_lookup, web_search)])

3 tools ready: ['policy_search', 'order_lookup', 'web_search']


In [13]:
# @title Step 10: Build and Run the Deep Agent
from deepagents import create_deep_agent
from IPython.display import Markdown, display

task = 'Order 10245 was delivered late and one item was missing. Tell the customer exactly where each refund stands and what the policy says a Gold member is owed for a 29-minute delay.' # @param {type:"string"}

AGENT_PROMPT = f"""You are a Blinkit customer-support AI agent.

You have three tools: policy_search (the rules), order_lookup (the database), web_search (the live web).

How to work:
1. Look up the customer's actual order and its line items before you say anything about it.
2. Check the policy before you promise a refund, a timeline or an exception.
3. Use more than one tool when the answer depends on both data and rules.
4. End with what you found, what the policy says, and what happens next.

Never invent an order, an item, an amount, a status or a policy line. If a tool returns nothing, say so.

SQLite schema available to order_lookup:
{SCHEMA}
"""

support_agent = create_deep_agent(
    model=f"google_genai:{MODEL}",
    tools=[policy_search, order_lookup, web_search],
    system_prompt=AGENT_PROMPT,
)

result = support_agent.invoke({"messages": [{"role": "user", "content": task}]})

print("--- TOOLS THE AGENT CHOSE ---")
for m in result["messages"]:
    for tc in (getattr(m, "tool_calls", None) or []):
        print(" *", tc["name"], "->", str(tc["args"])[:120])

display(Markdown(result["messages"][-1].content[0]["text"]))

--- TOOLS THE AGENT CHOSE ---
 * order_lookup -> {'sql': 'SELECT o.order_id, o.customer_id, o.delay_minutes, o.order_status, c.membership_tier FROM orders o JOIN custome
 * order_lookup -> {'sql': 'SELECT oi.order_item_id, p.product_name, oi.item_status, r.refund_status, r.amount_inr, r.reason, r.refund_id F
 * policy_search -> {'query': 'refund for late delivery Gold member delay minutes'}
 * order_lookup -> {'sql': 'SELECT order_id, delivery_fee_inr, surge_fee_inr, order_total_inr FROM orders WHERE order_id = 10245;'}
 * order_lookup -> {'sql': 'SELECT * FROM refunds WHERE order_id = 10245;'}


I have reviewed your order 10245 and the relevant policy regarding your Gold membership benefits.

### Current Refund Status
*   **Missing Item (Curd):** Your refund for the missing Curd (INR 36.00) has been processed and is **COMPLETED** (Refund ID: RFD300246).
*   **Late Delivery:** A goodwill credit for the late delivery is currently being processed. You have been issued a credit of **INR 150.00** (Refund ID: RFD300245), which is currently in **PROCESSING** status.

### Policy Regarding Your Gold Membership and Delay
For a Gold member, our policy states:
*   **Goodwill Credit:** For delays beyond 15 minutes, Gold members are entitled to the higher band of goodwill credit, which ranges from INR 100 to INR 150. You have been granted the maximum credit of INR 150.00.
*   **Fee Refund:** Our policy states that if a delivery is delayed by **more than 30 minutes**, the delivery fee and surge fee are refunded in full. As your order was delayed by exactly **29 minutes**, it does not meet the 30-minute threshold required for the additional mandatory refund of the surge fee (INR 25.00).

Since your goodwill credit for the delay is already in the processing stage, you should see the funds returned to your source payment method shortly. Please let me know if there is anything else I can assist you with.

**What changed:** nobody told it to query the order, then the line items, then the policy. It decided,
in that order, because the goal required all three.

One agent, one customer, one conversation. Now scale that to the whole operation.

## Stage 3 test prompts — AI Agent

Paste into the `task` box in Step 10. You no longer name the source; the agent picks.

**Works — one goal, multiple tools, an actual decision**

| # | Prompt | Tools it should reach for |
|---|---|---|
| 1 | Order 10245 was delivered late and one item was missing. Tell the customer where each refund stands and what the policy says they are owed. | order_lookup + policy_search |
| 2 | Keerthana R. is asking why her refund is taking so long. Check her history and answer her. | order_lookup |
| 3 | A Gold member wants compensation for a 29-minute delay on order 10245. Give me a decision with a reason and the amount. | policy_search + order_lookup |
| 4 | Has any completed refund breached our published 5-working-day timeline? How many? | both |
| 5 | Which Coimbatore dark store has the worst refund rate, and what does the policy let us do about it? | order_lookup + policy_search |
| 6 | Eggs keep going missing from bags. How often, from which stores, and is the customer entitled to a refund without a photo? | order_lookup + policy_search |
| 7 | A customer says the rider left an order containing cigarettes at the door. Look up whether any such order exists and give me the policy position. | both |
| 8 | How many tickets are at escalation stage 2 or 3, and what response time does the policy commit to for each? | order_lookup + policy_search |
| 9 | Which refund reason costs us the most money, and which rejection reason do we use most? Is each refusable under policy? | order_lookup + policy_search |
| 10 | Are quick-commerce platforms raising handling fees across the industry, and how does our own handling fee compare? | web_search + order_lookup |
| 11 | We cancelled 685 orders for stock-outs. Which SKUs are at zero stock in the most stores right now, and what do we owe those customers? | order_lookup + policy_search |

**Breaks — one agent, one context window**

| # | Prompt | Why it strains |
|---|---|---|
| 12 | Review the whole refund backlog, find root causes by store and SKU, draft customer messages and store warnings, and give me a board summary. | Four workstreams in one context. It drops threads. |
| 13 | Run a quality audit across all 60 cities and produce one report per zone. | Too many parallel investigations. |
| 14 | Investigate rising missing-item refunds, separately investigate rider shortage, then combine both. | Needs two independent lines of work held apart. |
| 15 | Actually credit INR 150 to the customer's wallet. | No write tool exists. Deliberately — money movement stays human. |

Prompt 1 is the demo. Watch the tool trace: it queries the order, then the line items, then reads the policy.
Nobody told it that order.

# 4. Agentic AI

One agent handles one request. Agentic AI handles a **goal** — by planning, splitting the work
across specialists, and reporting back.

A Deep Agent gets planning and delegation built in. We add the specialists.

In [14]:
# @title Step 11: Define the Specialist Subagents
refund_agent = {
    "name": "refund-agent",
    "description": "Investigates refund backlogs, reject reasons and refund eligibility against policy.",
    "system_prompt": (
        "You handle refunds. Query the database for refunds by status, reason, age, store and customer tier, "
        "and check policy_search for what the customer is actually entitled to. Blinkit refunds per item, "
        "so join order_items and products when a reason is item-level. Report counts and rupee amounts, never guesses."
    ),
    "tools": [order_lookup, policy_search],
}

quality_agent = {
    "name": "quality-agent",
    "description": "Finds dark stores, SKUs, cities and delivery patterns driving complaints, stock-outs and refunds.",
    "system_prompt": (
        "You analyse operational quality. Use order_lookup to find late-delivery rates, item-level failure rates "
        "(MISSING, DAMAGED, EXPIRED, WEIGHT_SHORT), stock-out cancellations and low ratings by city, store and SKU. "
        "Always report the denominator alongside the rate, and ignore stores with fewer than 30 orders."
    ),
    "tools": [order_lookup],
}

comms_agent = {
    "name": "comms-agent",
    "description": "Drafts customer apology messages and dark-store or partner warning notices.",
    "system_prompt": (
        "You write short, plain, professional messages in Indian English. No emojis. "
        "Use only the facts and figures handed to you. Never promise a refund the policy does not allow, "
        "and never promise anything about a delivery partner's employment."
    ),
    "tools": [policy_search],
}

print("Subagents:", [a["name"] for a in (refund_agent, quality_agent, comms_agent)])

Subagents: ['refund-agent', 'quality-agent', 'comms-agent']


In [16]:
# @title Step 12: Run the Supervisor — Agentic AI
from deepagents import create_deep_agent
from IPython.display import Markdown, display

goal = 'Our refund payout is rising. Find which cities, dark stores and SKUs are driving it, check what the policy lets us refuse, and draft one apology mail for the affected Gold customers.' # @param {type:"string"}

SUPERVISOR_PROMPT = f"""You are the Blinkit support operations supervisor.

You do not do the analysis yourself. You write a plan, delegate each step to the right specialist,
and combine what they return into one report for the operations head.

Your specialists: refund-agent, quality-agent, comms-agent.

Always finish with a short recommendation list with real numbers attached to each recommendation.

SQLite schema available to your specialists:
{SCHEMA}
"""

ops_supervisor = create_deep_agent(
    model=f"google_genai:{MODEL}",
    tools=[order_lookup, policy_search, web_search],
    subagents=[refund_agent, quality_agent, comms_agent],
    system_prompt=SUPERVISOR_PROMPT,
)

result = ops_supervisor.invoke(
    {"messages": [{"role": "user", "content": goal}]},
    {"recursion_limit": 50},
)

print("--- PLAN AND DELEGATION ---")
for m in result["messages"]:
    for tc in (getattr(m, "tool_calls", None) or []):
        print(" *", tc["name"], "->", str(tc["args"])[:140])

display(Markdown(result["messages"][-1].content[0]["text"]))

--- PLAN AND DELEGATION ---
 * task -> {'description': '1. Identify the top 5 cities and top 10 dark stores with the highest total refund amount in the last 30 days.\n2. Identify 
 * task -> {'description': 'Search the internal policy for criteria on when we can refuse refunds (specifically regarding item condition, time windows,
 * task -> {'subagent_type': 'comms-agent', 'description': 'Draft an apology email for Gold customers who have received damaged or incorrect items.\nTo


The recent audit of refund payouts over the last 30 days reveals significant operational hotspots. We are currently seeing high refund volumes concentrated in major urban centers and specific fresh produce categories.

### Summary of Findings
*   **Top Refund Cities:** Pune, Kolkata, and Gurugram account for the highest total refund amounts.
*   **Key Dark Store Hotspots:** The "Blinkit Park Street Store" and "Blinkit Aundh Store" have the highest refund volumes and should be prioritised for immediate inspection.
*   **Top Problematic SKUs:** Fresh produce and perishables—specifically **Eggs, Carrots, Apples, and Potatoes**—dominate our refund frequency metrics.

### Policy Enforcement Strategy
We are currently under-utilizing our existing refund refusal criteria. We should reinforce training for support agents to ensure:
1.  **Strict 48-hour claim window adherence** for general items.
2.  **Mandatory photo evidence** for all Damaged, Expired, Wrong Item, and Quality-related claims.
3.  **Cross-referencing with "Leave at door" logs** for claims filed long after delivery.

### Recommended Actions
1.  **Targeted Store Quality Audit:** Conduct an immediate quality audit at the **Blinkit Park Street Store** (highest refund store) to verify picking and packing accuracy for fresh produce (Priority: High).
2.  **Vendor Quality Review:** Review suppliers for **Eggs, Carrots, and Apples**, as these SKUs drive the highest frequency of quality-related refunds across multiple cities (Target: 10% reduction in refund rate).
3.  **Standardized Rejection Protocols:** Implement a stricter "No photo, no refund" policy for DAMAGED/EXPIRED claims, which should result in an estimated **5-8% decrease in total refund payouts** (Priority: Medium).
4.  **Gold Customer Engagement:** Use the drafted apology mail for identified impacted Gold customers to preserve loyalty while we remediate store-level issues.

## Stage 4 test prompts — Agentic AI

Paste into the `goal` box in Step 12. These are goals, not questions.

**Works — plan, delegate, combine**

| # | Prompt |
|---|---|
| 1 | Our refund payout is rising. Find which cities, dark stores and SKUs are driving it, check what the policy lets us refuse, and draft one apology for affected Gold customers. |
| 2 | Prepare a Monday operations review: refund backlog, worst-performing dark stores, top stock-out SKUs, and the three actions you recommend. |
| 3 | Audit whether we are honouring our own refund policy. Compare promised timelines against actual by refund mode, and list every breach. |
| 4 | Build the case for pausing our worst dark store: the numbers, the policy grounds, and the warning notice to the store manager. |
| 5 | Our Gold members are complaining. Find out whether Gold customers actually get faster delivery and faster support than Regular ones, then draft the response. |
| 6 | Which ticket category has the worst CSAT, why, and what should we change? |
| 7 | Give me a city scorecard for the South zone: late rate, refund cost, stock-out cancellations, and one recommendation per city. |
| 8 | Eggs, milk and lemons keep failing in the bag. Quantify it by store, check the policy on each failure type, and write a packing SOP reminder for pickers. |
| 9 | We rejected a number of refunds for NO_PHOTO. Find how many, check whether the policy actually required a photo for each reason, and draft an appeal response template. |
| 10 | We are opening in a new tier-3 city. Look at how our existing tier-3 cities perform on late rate and stock-outs, and tell me what to fix before launch. |
| 11 | Monsoon is coming. Compare late rates and surge-fee refunds in rain-zone cities against the rest, and recommend a rider-staffing plan. |

**Should not — the limits worth saying out loud**

| # | Prompt | Why the answer is no |
|---|---|---|
| 12 | Delete the negative reviews for our worst dark store. | Section 9 of the policy forbids it. A good agent refuses and cites it. |
| 13 | Release the INR 1,55,341 of pending refunds now. | No money-movement tool. Financial actions need a human approver. |
| 14 | Tell me which delivery partner to terminate. | Section 9: a human decision about a named individual. Escalate, do not automate. |
| 15 | Predict next quarter's refund payout. | No forecasting tool wired in. It will estimate, and sound confident doing it. |

Prompts 12 to 15 are worth running anyway. An agent that refuses well is more impressive to a
business audience than one that says yes to everything.

## What the audience just watched

| Step | Level | The question it could finally answer |
|---|---|---|
| 3, 4 | LLM | "What is a handling fee?" — and an honest "I cannot check that" |
| 6, 7, 8 | RAG | "Where is *my* INR 150, and what does the policy say about my missing curd?" |
| 10 | AI Agent | "Sort out order 10245 for this customer." |
| 12 | Agentic AI | "Our refund cost is rising — find out why, by store and by SKU, and fix it." |

Each level did not replace the one before it. Level 4 still contains a language model, a
retriever and a set of tools. It just stopped needing you to say which one to use.

**If the model struggles with tool calls:** change `MODEL` in Step 1 to a full Flash model. The
Lite model is fast and free, but it plans less reliably across twelve tool calls.

**If SQL RAG writes a bad query:** the schema has 11 tables. Ask the question with the table in mind
("from order_items, which product ...") and the Lite model gets it right far more often.